In [1]:
# 1. Install the modern tools we actually need first
!pip install huggingface-hub==0.24.0 transformers==4.40.0 accelerate==0.31.0

# 2. Force install SummaC without checking its dependencies
!pip install summac --no-deps

# 3. Manually install the other things SummaC needs that won't cause conflicts
!pip install nltk scipy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.0/419.0 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 128.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installation

In [2]:
from google.colab import drive
import os

In [3]:
#  Mount Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Paths
source_sample_dir = 'YOUR_PATH'
output_dir = 'YOUR_PATH'

all_sources = []
all_summaries = []
filenames = []

# 3. Load the file pairs
file_list = sorted([f for f in os.listdir(output_dir) if f.endswith('.txt')])

for file_name in file_list:
    # Remove "_summary" from filenames
    src_file = file_name.replace("_summary", "")

    src_path = os.path.join(source_sample_dir, src_file)
    sum_path = os.path.join(output_dir, file_name)

    if os.path.exists(src_path):
        with open(src_path, 'r', encoding='utf-8') as f:
            all_sources.append(f.read())
        with open(sum_path, 'r', encoding='utf-8') as f:
            all_summaries.append(f.read())
        filenames.append(file_name)

print(f" Loaded {len(all_summaries)} Sinhala file pairs successfully.")

 Loaded 242 Sinhala file pairs successfully.


In [5]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
from summac.model_summac import SummaCConv
import numpy as np
import pandas as pd

print("Initializing SummaC Conv model...")

# Initialize the Convolutional SummaC model
#  device="cuda" to leverage the T4 GPU
model_conv = SummaCConv(models=["vitc"], bin_number=10, aggregation="mean", device="cuda")

# Calculate scores
print("Calculating factual consistency scores...")
summac_res = model_conv.score(all_sources, all_summaries)

# Create a results table
summac_results = pd.DataFrame({
    "File": filenames,
    "SummaC_Score": summac_res["scores"]
})

# Save results to CSV
output_path = 'YOUR_PATH'
summac_results.to_csv(output_path, index=False)

average_score = np.mean(summac_res["scores"])

print("\n" + "="*30)
print(f" FINAL AVERAGE SUMMAC SCORE: {average_score:.4f}")
print(f"Results saved to: {output_path}")
print("="*30)

Initializing SummaC Conv model...
Calculating factual consistency scores...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/217 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/235M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/summac/model_summac.py:298: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  histograms = torch.FloatTensor(histograms).to(self.device)



 FINAL AVERAGE SUMMAC SCORE: 0.7520
Results saved to: /content/drive/MyDrive/SinBrief_Implementation/Evaluations/Llama/Result_Sheets/summac_results.csv
